# NPR Model Evaluation (Baseline)

## Overview

**NPR 모델만으로 Corrupted Dataset 평가** (NRAM 없이, Training 없이 순수 Evaluation만)

**목적:**
- NPR 모델의 Baseline 성능 측정
- 다양한 corruption/severity에 대한 robustness 평가

## 1. Import

In [1]:
import sys
# Clear cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['NPR', 'npr', 'LGrad', 'lgrad', 'nram']):
        del sys.modules[mod]

In [2]:
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

# Dataset and metrics
from utils.data.dataset import CorruptedDataset
from utils.eval.metrics import PredictionCollector, MetricsCalculator

# NPR Model
from model.NPR.npr_model import NPR

In [3]:
!nvidia-smi

Tue Jan 20 13:58:08 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:04:00.0 Off |                    0 |
| N/A   37C    P0              27W / 250W |      4MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 2. Configuration

In [4]:
# Device
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Datasets - using new corrupted_dataset structure
DATASETS = ["ADM", "DDPM", "IDDPM", "LDM", "PNDM", "VQDIFFUSION", "SDV1", "SDV2", "PROGAN", "STYLEGAN", "STYLEGAN2", "BIGGAN", "CYCLEGAN", "STARGAN", "GAUGAN", "DEEPFAKE"]

# Corruptions and Severities
CORRUPTIONS = ["color_contrast", "color_saturation", "resize", "gaussian_blur"]
SEVERITIES = ["corrupted1", "corrupted2", "corrupted3", "corrupted4", "corrupted5"]

# Paths
DATA_ROOT = "corrupted_dataset"

BATCH_SIZE = 16

Using device: cuda:0


## 3. Load NPR Model

In [5]:
NPR_WEIGHTS = "model/NPR/weights/NPR.pth"

model = NPR(
    weights=NPR_WEIGHTS,
    device=DEVICE
)

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"NPR model loaded")

/workspace/robust_deepfake_ai/model/NPR/npr_model.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weights, map_location="cpu")


NPR model loaded


## 4. Create Dataset

In [6]:
dataset = CorruptedDataset(
    root=DATA_ROOT,
    datasets=DATASETS,
    corruptions=CORRUPTIONS,
    severities=SEVERITIES,
    transform=transform
)

print(f"Total samples: {len(dataset)}")
print(f"Datasets: {DATASETS}")
print(f"Corruptions: {CORRUPTIONS}")
print(f"Severities: {SEVERITIES}")

Total samples: 1852120
Datasets: ['ADM', 'DDPM', 'IDDPM', 'LDM', 'PNDM', 'VQDIFFUSION', 'SDV1', 'SDV2', 'PROGAN', 'STYLEGAN', 'STYLEGAN2', 'BIGGAN', 'CYCLEGAN', 'STARGAN', 'GAUGAN', 'DEEPFAKE']
Corruptions: ['color_contrast', 'color_saturation', 'resize', 'gaussian_blur']
Severities: ['corrupted1', 'corrupted2', 'corrupted3', 'corrupted4', 'corrupted5']


## 5. Evaluation Function

In [7]:
def evaluate_model(model, dataloader, device, name="test"):
    """
    Evaluate NPR model (no TTA, just inference).
    """
    model.eval()
    collector = PredictionCollector()
    calc = MetricsCalculator()

    pbar = tqdm(dataloader, desc=name)
    with torch.no_grad():
        for batch in pbar:
            images, labels, metadata = batch
            images = images.to(device)

            logits = model(images)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            probs = torch.sigmoid(logits).squeeze(1)
            collector.update(labels, probs.cpu(), threshold=0.5)

    metrics = calc.compute_from_collector(collector, name=name)
    return metrics

## 6. Full Evaluation

In [8]:
# Evaluate on all combinations
calc = MetricsCalculator()
all_results = {}  # {(dataset, corruption, severity): metrics}
corruption_avg_results = {}  # {(dataset, corruption): averaged_metrics}

for dataset_name in DATASETS:
    print(f"\n{'#'*70}")
    print(f"# Dataset: {dataset_name}")
    print(f"{'#'*70}")
    
    for corruption in CORRUPTIONS:
        severity_results = []  # Store results for severity 1-5
        
        print(f"\n{'='*70}")
        print(f"Corruption: {corruption}")
        print(f"{'='*70}")
        
        for severity in SEVERITIES:
            # Get indices for this specific combination
            indices = [
                i for i, s in enumerate(dataset.samples)
                if s['dataset'] == dataset_name 
                and s['corruption'] == corruption 
                and s['severity'] == severity
            ]
            
            if len(indices) == 0:
                print(f"  {severity}: No samples, skipping")
                continue
            
            print(f"\n  [{severity}] Samples: {len(indices)}")
            
            # Create dataloader
            dataloader = DataLoader(
                Subset(dataset, indices),
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=4,
                drop_last=True
            )
            
            # Evaluate
            metrics = evaluate_model(
                model=model,
                dataloader=dataloader,
                device=DEVICE,
                name=f"{corruption}-{severity}"
            )
            
            # Print results for this severity
            print(f"      Acc: {metrics['accuracy']*100:.2f}%  AUC: {metrics['auc']*100:.2f}%  AP: {metrics['ap']*100:.2f}%  F1: {metrics['f1']*100:.2f}%")
            
            # Store results
            all_results[(dataset_name, corruption, severity)] = metrics
            severity_results.append(metrics)
        
        # Compute average across severities 1-5 for this corruption
        if severity_results:
            avg_metrics = {
                'accuracy': np.mean([m['accuracy'] for m in severity_results]),
                'auc': np.mean([m['auc'] for m in severity_results]),
                'ap': np.mean([m['ap'] for m in severity_results]),
                'f1': np.mean([m['f1'] for m in severity_results]),
            }
            corruption_avg_results[(dataset_name, corruption)] = avg_metrics
            
            print(f"\n  >> {corruption} Average (severity 1-5):")
            print(f"      Acc: {avg_metrics['accuracy']*100:.2f}%  AUC: {avg_metrics['auc']*100:.2f}%  AP: {avg_metrics['ap']*100:.2f}%  F1: {avg_metrics['f1']*100:.2f}%")

# ============================================================
# Summary Tables
# ============================================================
print(f"\n\n{'='*80}")
print(f"RESULTS SUMMARY (NPR Baseline - No NRAM, No TTA)")
print(f"{'='*80}")

# Table 1: Detailed results per severity
print(f"\n[Table 1] Detailed Results (per severity)")
print(f"{'Dataset':<15} {'Corruption':<20} {'Severity':<12} {'Accuracy':<10} {'AUC':<10} {'AP':<10} {'F1':<10}")
print("-" * 87)
for (dataset_name, corruption, severity), metrics in all_results.items():
    print(f"{dataset_name:<15} {corruption:<20} {severity:<12} {metrics['accuracy']*100:>6.2f}%    {metrics['auc']*100:>6.2f}%    {metrics['ap']*100:>6.2f}%    {metrics['f1']*100:>6.2f}%")

# Table 2: Averaged results per corruption (main result)
print(f"\n\n[Table 2] Averaged Results (severity 1-5 mean) - MAIN RESULT")
print(f"{'Dataset':<15} {'Corruption':<25} {'Accuracy':<10} {'AUC':<10} {'AP':<10} {'F1':<10}")
print("-" * 80)
for (dataset_name, corruption), metrics in corruption_avg_results.items():
    print(f"{dataset_name:<15} {corruption:<25} {metrics['accuracy']*100:>6.2f}%    {metrics['auc']*100:>6.2f}%    {metrics['ap']*100:>6.2f}%    {metrics['f1']*100:>6.2f}%")

# Overall average
overall_avg = {
    'accuracy': np.mean([m['accuracy'] for m in corruption_avg_results.values()]),
    'auc': np.mean([m['auc'] for m in corruption_avg_results.values()]),
    'ap': np.mean([m['ap'] for m in corruption_avg_results.values()]),
    'f1': np.mean([m['f1'] for m in corruption_avg_results.values()]),
}
print("-" * 80)
print(f"{'Overall Average':<40} {overall_avg['accuracy']*100:>6.2f}%    {overall_avg['auc']*100:>6.2f}%    {overall_avg['ap']*100:>6.2f}%    {overall_avg['f1']*100:>6.2f}%")


######################################################################
# Dataset: ADM
######################################################################

Corruption: color_contrast

  [corrupted1] Samples: 7000


color_contrast-corrupted1: 100%|██████████| 437/437 [00:09<00:00, 44.02it/s]


      Acc: 80.15%  AUC: 99.86%  AP: 99.98%  F1: 86.90%

  [corrupted2] Samples: 7000


color_contrast-corrupted2: 100%|██████████| 437/437 [00:08<00:00, 48.87it/s]


      Acc: 82.47%  AUC: 99.93%  AP: 99.99%  F1: 88.60%

  [corrupted3] Samples: 7000


...

      Acc: 49.80%  AUC: 50.04%  AP: 49.81%  F1: 66.49%

  [corrupted4] Samples: 5405


gaussian_blur-corrupted4: 100%|██████████| 337/337 [00:07<00:00, 46.35it/s]


      Acc: 49.80%  AUC: 50.02%  AP: 49.81%  F1: 66.49%

  [corrupted5] Samples: 5405


gaussian_blur-corrupted5: 100%|██████████| 337/337 [00:07<00:00, 46.69it/s]

      Acc: 49.80%  AUC: 50.00%  AP: 49.80%  F1: 66.49%

  >> gaussian_blur Average (severity 1-5):
      Acc: 49.77%  AUC: 49.98%  AP: 49.78%  F1: 66.46%


RESULTS SUMMARY (NPR Baseline - No NRAM, No TTA)

[Table 1] Detailed Results (per severity)
Dataset         Corruption           Severity     Accuracy   AUC        AP         F1        
---------------------------------------------------------------------------------------
ADM             color_contrast       corrupted1    80.15%     99.86%     99.98%     86.90%
ADM             color_contrast       corrupted2    82.47%     99.93%     99.99%     88.60%
ADM             color_contrast       corrupted3    86.41%     99.98%    100.00%     91.39%
ADM             color_contrast       corrupted4    90.59%     99.99%    100.00%     94.19%
ADM             color_contrast       corrupted5    95.01%     99.99%    100.00%     97.00%
ADM             color_saturation     corrupted1    85.24%     99.83%     99.97%     90.58%
ADM             color_sa

## Summary

이 노트북은 **NPR 모델의 Baseline 성능**을 측정합니다.

- NRAM 모듈 없음
- Test-Time Adaptation 없음
- 순수 inference만 수행

이 결과를 NRAM이 적용된 결과와 비교하여 NRAM의 효과를 확인할 수 있습니다.